# IMDB 情感分类：BiLSTM + Attention

使用 PyTorch 完成 IMDB 影评二分类。模型结构为 Embedding → BiLSTM → Attention Pooling → LayerNorm → 三层全连接分类器。

运行前请先按照 `data/README.md` 下载并解压预处理数据。原始实验记录的最佳验证结果为 **12109/13636（88.80%）**；由于硬件与 PyTorch 版本差异，重新训练结果可能略有波动。

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset

torch.__version__

In [ ]:
MAX_WORDS = 25000  # 保留原始实验配置；当前数据中的最大 token ID 为 9999
MAX_LEN = 200
BATCH_SIZE = 256
EMB_SIZE = 256   # embedding size
HID_SIZE = 256   # lstm hidden size
DROPOUT = 0.50
EPOCH_NUMBER = 40
LEARNING_RATE = 1e-3
PATIENCE = 8
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
required_files = ['x_train.npy', 'y_train.npy', 'x_val.npy', 'y_val.npy']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f'Missing data files: {missing_files}. Follow {DATA_DIR / "README.md"} to download them.'
    )

x_train = np.load(DATA_DIR / 'x_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
x_val = np.load(DATA_DIR / 'x_val.npy')
y_val = np.load(DATA_DIR / 'y_val.npy')

print('train:', x_train.shape, np.bincount(y_train))
print('validation:', x_val.shape, np.bincount(y_val))
print('max token id:', int(max(x_train.max(), x_val.max())))
assert x_train.shape[1] == x_val.shape[1] == MAX_LEN
assert int(max(x_train.max(), x_val.max())) < MAX_WORDS

In [ ]:
# 转化为TensorDataset
train_data = TensorDataset(torch.LongTensor(x_train), torch.LongTensor(y_train))
val_data = TensorDataset(torch.LongTensor(x_val), torch.LongTensor(y_val))

In [ ]:
# 转化为 DataLoader
train_sampler = RandomSampler(train_data)
train_loader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE, pin_memory=torch.cuda.is_available())

val_sampler = SequentialSampler(val_data)
val_loader = DataLoader(val_data, sampler=val_sampler, batch_size=BATCH_SIZE, pin_memory=torch.cuda.is_available())

In [ ]:
# 定义RNN模型用于文本分类
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_output):
        attention_scores = self.attention(lstm_output).squeeze(-1)
        attention_weights = F.softmax(attention_scores, dim=1)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), lstm_output).squeeze(1)
        return context_vector, attention_weights


class Model(nn.Module):
    def __init__(self, max_words, emb_size, hid_size, dropout):
        super(Model, self).__init__()
        self.max_words = max_words
        self.emb_size = emb_size
        self.hid_size = hid_size
        self.dropout = dropout
        self.Embedding = nn.Embedding(self.max_words, self.emb_size)
        self.RNN = nn.LSTM(
            self.emb_size,
            self.hid_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.0  # 单层 LSTM 不使用内部 dropout；外部 self.dp 仍为 0.5
        )
        self.dp = nn.Dropout(self.dropout)
        self.attention = Attention(self.hid_size * 2)
        self.norm = nn.LayerNorm(self.hid_size * 2)
        self.fc1 = nn.Linear(self.hid_size * 2, self.hid_size)
        self.fc2 = nn.Linear(self.hid_size, self.hid_size // 2)
        self.fc3 = nn.Linear(self.hid_size // 2, 2)
    
    def forward(self, x, return_attention=False):
        """
        input : [bs, maxlen]
        output: [bs, 2] 
        """
        x = self.Embedding(x)  # [bs, ml, emb_size]
        x = self.dp(x)
        x, _ = self.RNN(x)  # [bs, ml, 2*hid_size]
        x = self.dp(x)
        x, attention_weights = self.attention(x)
        x = self.norm(x)
        x = F.relu(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dp(x)
        x = self.fc2(x)
        x = self.dp(x)
        x = self.fc3(x)
        out = F.log_softmax(x, dim=1)
        if return_attention:
            return out, attention_weights
        return out  # [bs, 2]

In [ ]:
def train(model, device, train_loader, optimizer, epoch):   # 训练模型
    model.train()
    criterion = nn.NLLLoss()
    total_loss = 0.0
    correct = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        log_probs = model(x)
        loss = criterion(log_probs, y)  # 得到loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (log_probs.argmax(dim=1) == y).sum().item()
        if(batch_idx + 1) % 10 == 0:    # 打印loss
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(x), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    return total_loss / len(train_loader.dataset), correct / len(train_loader.dataset)

In [ ]:
@torch.no_grad()
def test(model, device, test_loader):    # 测试模型
    model.eval()
    criterion = nn.NLLLoss(reduction='sum')  # 累加loss
    val_loss = 0.0 
    acc = 0 
    for batch_idx, (x, y) in enumerate(test_loader):
        x, y = x.to(device), y.to(device)
        log_probs = model(x)
        val_loss += criterion(log_probs, y).item()
        pred = log_probs.max(-1, keepdim=True)[1]   # .max() 2输出，分别为最大值和最大值的index
        acc += pred.eq(y.view_as(pred)).sum().item()    # 记得加item()
    val_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)'.format(
        val_loss, acc, len(test_loader.dataset),
        100. * acc / len(test_loader.dataset)))
    return acc / len(test_loader.dataset)

In [ ]:
model = Model(MAX_WORDS, EMB_SIZE, HID_SIZE, DROPOUT).to(DEVICE)
print(model)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_acc = 0.0 
bad_epoch = 0
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PATH = MODEL_DIR / 'model.pth'

for epoch in range(1, EPOCH_NUMBER + 1): 
    train_loss, train_acc = train(model, DEVICE, train_loader, optimizer, epoch)
    acc = test(model, DEVICE, val_loader)
    if best_acc < acc: 
        best_acc = acc 
        bad_epoch = 0
        torch.save(model.state_dict(), PATH)
    else:
        bad_epoch += 1
    print("acc is: {:.4f}, best acc is {:.4f}\n".format(acc, best_acc))
    if bad_epoch >= PATIENCE:
        print("Early stopping at epoch {}".format(epoch))
        break

In [ ]:
# 检验保存的模型
best_model = Model(MAX_WORDS, EMB_SIZE, HID_SIZE, DROPOUT).to(DEVICE)
best_model.load_state_dict(torch.load(PATH, map_location=DEVICE, weights_only=True))
test(best_model, DEVICE, val_loader)